In [1]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

Project Root: /home/ayon/git/EventCameraProject


In [2]:
import torch

from src.losses.latent_consistency_loss import LatentConsistencyLoss
from src.losses.depth_smoothness_loss import DepthSmoothnessLoss
from src.losses.pose_temporal_consistency_loss import PoseTemporalConsistencyLoss
from src.losses.depth_temporal_consistency_loss import (
    DepthTemporalConsistencyLoss,
)

from src.losses.dynamic_mask_regularization_loss import DynamicMaskRegularizationLoss

from src.models.world_model.model import WorldModel

from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

# ---------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

print("Device:", device)

# ---------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

# ---------------------------------------------------------------------
# Transform Pipeline
# ---------------------------------------------------------------------

transform = Compose(
    [
        ToTensor(),
        NormalizeEventTime(),
        NormalizeIMU(),
        VoxelizeEvents(
            num_bins=5,
        ),
    ]
)

# ---------------------------------------------------------------------
# Get one batch
# ---------------------------------------------------------------------

raw_batch = next(iter(loader))

voxel_batch = transform(raw_batch)

# ---------------------------------------------------------------------
# Build Event Tensor
# ---------------------------------------------------------------------

voxels = torch.stack(
    [
        frame.voxel_grid
        for frame in voxel_batch.frames
    ],
    dim=1,
).to(device)

print()
print("=" * 90)
print("EVENT VOXELS")
print("=" * 90)
print("Shape :", tuple(voxels.shape))
print("dtype :", voxels.dtype)
print("device:", voxels.device)

assert voxels.ndim == 5

batch_size, sequence_length, num_bins, height, width = voxels.shape


Device: cpu
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

EVENT VOXELS
Shape : (2, 4, 5, 480, 640)
dtype : torch.float32
device: cpu


In [ ]:
# ==========================================================
# MODEL + LOSS INTEGRATION TEST
# ==========================================================

print()
print("=" * 90)
print("WORLD MODEL + LOSS TEST")
print("=" * 90)


# ----------------------------------------------------------
# Model
# ----------------------------------------------------------

model = WorldModel().to(device)

model.train()


# ----------------------------------------------------------
# Losses
# ----------------------------------------------------------

latent_loss_fn = LatentConsistencyLoss()

depth_smoothness_loss_fn = DepthSmoothnessLoss()

pose_temporal_consistency_loss_fn = PoseTemporalConsistencyLoss()

depth_temporal_consistency_loss_fn = (
    DepthTemporalConsistencyLoss()
)

dynamic_mask_regularization_loss_fn = (
    DynamicMaskRegularizationLoss()
)

# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

outputs = model(
    voxels,
    voxel_batch,
)


# ----------------------------------------------------------
# Target latent
#
# Current latent encoded directly from the current frame.
# This serves as the supervision target.
# ----------------------------------------------------------

target_state = outputs["temporal_features"][:, -1]


# ----------------------------------------------------------
# Compute losses
# ----------------------------------------------------------

latent_losses = latent_loss_fn(

    predicted_state=outputs["predicted_state"],

    rendered_state=outputs["rendered_state"],

    target_state=target_state,

)


depth_loss = depth_smoothness_loss_fn(
    outputs["depth"]
)


pose_temporal_consistency_loss_fn = pose_temporal_consistency_loss_fn(
    outputs["poses"]
)

depth_temporal_loss = (
    depth_temporal_consistency_loss_fn(
        outputs["depths"]
    )
)

dynamic_mask_regularization_loss = dynamic_mask_regularization_loss_fn(
    outputs["mask"],
)

# ----------------------------------------------------------
# Total loss
#
# Future losses will simply be added here.
# ----------------------------------------------------------

total_loss = (

    latent_losses["loss"]

    +

    depth_loss["loss"]

    +

    pose_temporal_consistency_loss_fn["loss"]

    +

    depth_temporal_loss["loss"]
    +
    dynamic_mask_regularization_loss["loss"]

)


# ----------------------------------------------------------
# Print
# ----------------------------------------------------------

print()

print()
print("-"*90)
print("Loss Breakdown")
print("-"*90)

print(f"Prediction Loss      : {latent_losses['prediction_loss']:.6f}")
print(f"Rendering Loss       : {latent_losses['rendering_loss']:.6f}")
print(f"Agreement Loss       : {latent_losses['agreement_loss']:.6f}")

print()

print(f"Depth Smoothness     : {depth_loss['loss']:.6f}")
print(f"Depth Temporal       : {depth_temporal_loss['loss']:.6f}")

print()

print(f"Pose Temporal        : {pose_temporal_consistency_loss_fn['loss']:.6f}")

print()

print(f"Mask Sparsity        : {dynamic_mask_regularization_loss['sparsity_loss']:.6f}")
print(f"Mask Confidence      : {dynamic_mask_regularization_loss['confidence_loss']:.6f}")
print(f"Dynamic Ratio        : {dynamic_mask_regularization_loss['dynamic_ratio']:.4f}")

print()
print(f"TOTAL LOSS           : {total_loss:.6f}")

# ----------------------------------------------------------
# Assertions
# ----------------------------------------------------------

assert torch.isfinite(total_loss)

assert total_loss.ndim == 0


# ----------------------------------------------------------
# Gradient Test
# ----------------------------------------------------------

model.zero_grad()

total_loss.backward()

num_grad = 0

for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(
            parameter.grad
        ).all()

        num_grad += 1


print()

print(
    "Parameters with gradients:",
    num_grad,
)

assert num_grad > 0


print()

print("=" * 90)
print("✓ WORLD MODEL + LOSS TEST PASSED")
print("=" * 90)


WORLD MODEL + LOSS TEST
jij


------------------------------------------------------------------------------------------
Loss Breakdown
------------------------------------------------------------------------------------------
Prediction Loss      : 0.156485
Rendering Loss       : 0.082330
Agreement Loss       : 0.153258

Depth Smoothness     : 0.220413
Depth Temporal       : 0.021059



TypeError: unsupported format string passed to dict.__format__

In [ ]:
from src.losses.total_loss import TotalLoss

total_loss_fn = TotalLoss(
    latent_weight=1.0,
    depth_smoothness_weight=1.0,
    pose_temporal_weight=1.0,
    depth_temporal_weight=1.0,
    dynamic_mask_weight=1.0,
)


total_loss = total_loss_fn(
    outputs=outputs
)

/home/ayon/git/EventCameraProject/src/losses/total_loss.py:309: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  * pose_temporal["loss"]


IndexError: too many indices for tensor of dimension 0

In [ ]:
import inspect

print(inspect.signature(TotalLoss.__init__))

(self, latent_weight: 'float' = 1.0, depth_smoothness_weight: 'float' = 1.0, pose_temporal_weight: 'float' = 1.0, depth_temporal_weight: 'float' = 1.0, dynamic_mask_weight: 'float' = 1.0)


In [ ]:
import src.losses.total_loss as total_loss_module

print(total_loss_module.__file__)
print(dir(total_loss_module))

/home/ayon/git/EventCameraProject/src/losses/total_loss.py
['DepthSmoothnessLoss', 'DepthTemporalConsistencyLoss', 'DynamicMaskRegularizationLoss', 'LatentConsistencyLoss', 'PoseTemporalConsistencyLoss', 'TotalLoss', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'nn', 'torch']
